In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/kaytiny/avito123/benchmark_queries.parquet
/kaggle/input/datasets/kaytiny/avito123/train.parquet
/kaggle/input/datasets/kaytiny/avito123/benchmark_items.parquet


In [2]:
!pip install -q rank-bm25

In [3]:
!pip install -q faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 72.4 MB/s eta 0:00:00:00:0100:01


In [4]:
!pip install -q sentence-transformers

In [5]:
!pip install bm25s PyStemmer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 752.4/752.4 kB 15.3 MB/s eta 0:00:0000:01


In [6]:

"""
Каналы -> RRF-склейка + бонус локации -> топ-50:
[BM25] + [dense (дообученный E5)] + [категорийный приор из train].
ВАЛИДАЦИЯ: после подбора recall пересчитывается по ВСЕМ val-запросам; срез каналов явный.
"""

import gc
import os
import numpy as np
import pandas as pd
import torch
import faiss
from collections import Counter
from scipy.sparse import csr_matrix
from sklearn.feature_extraction.text import CountVectorizer
from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader

os.environ.setdefault('CUDA_VISIBLE_DEVICES', '0')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
if torch.cuda.is_available():
    torch.cuda.empty_cache(); gc.collect()

DATA_DIR     = os.getenv('DATA_DIR', '/kaggle/input/datasets/kaytiny/avito123')
WORK_DIR     = '/kaggle/working'
BASE_MODEL   = os.getenv('BASE_MODEL', 'intfloat/multilingual-e5-base')
DEVICE       = 'cuda' if torch.cuda.is_available() else 'cpu'

TOP_K        = 50
CHANNEL_K    = 1000
VAL_FRAC     = 0.10
TUNE_QUERIES = 8000
SEED         = 42

TRAIN_BS     = int(os.getenv('TRAIN_BS', 32))
TRAIN_LR     = float(os.getenv('TRAIN_LR', 2e-5))
MAX_SEQ      = int(os.getenv('MAX_SEQ', 256))

FAST_STEPS     = int(os.getenv('FAST_STEPS', 2000))      # шагов fit на одну подвыборку
SKIP_DENSE_FIT = os.getenv('SKIP_DENSE_FIT', '0') == '1' # 1 = zero-shot (страховочный сабмит)
SAVE_MODEL     = os.getenv('SAVE_MODEL', '1') == '1'
LOAD_MODEL     = os.getenv('LOAD_MODEL', '') or None
USE_VAL        = os.getenv('USE_VAL', '1') == '1'        # 0 = без валидации/подбора (1 фит)

np.random.seed(SEED); torch.manual_seed(SEED)

# векторизация текста 
def norm_s(s: pd.Series) -> pd.Series:
    s = s.fillna('').astype(str).str.lower()
    s = s.str.replace(r'[^\w\s]', ' ', regex=True)
    s = s.str.replace(r'\s+', ' ', regex=True).str.strip()
    return s


def add_query_cols(df: pd.DataFrame) -> None:
    body = (norm_s(df['search_query']) + ' ' +
            norm_s(df['search_infm_params_text']) + ' ' +
            norm_s(df['search_category'])).str.replace(r'\s+', ' ', regex=True).str.strip()
    df['q_bow']  = body
    df['q_full'] = 'query: ' + body


def add_item_cols(df: pd.DataFrame, desc_chars: int = 400) -> None:
    desc = df['item_description_raw'].fillna('').astype(str).str[:desc_chars]
    body = (norm_s(df['item_category_id']) + ' ' +
            norm_s(df['item_microcat_id']) + ' ' +
            norm_s(df['item_title_raw']) + ' ' +
            norm_s(df['item_infm_params_text']) + ' ' +
            norm_s(desc)).str.replace(r'\s+', ' ', regex=True).str.strip()
    df['i_bow']  = body
    df['i_full'] = 'passage: ' + body

# BM25 по мешку слов объявления + предвычисленная sparse-матрица коэффициентов idf·(k1+1)·tf/(tf+K_d)
class BM25:
    def __init__(self, k1=1.5, b=0.75):
        self.k1, self.b = k1, b

    def fit(self, corpus_texts):
        self.vec = CountVectorizer(token_pattern=r'(?u)\b\w+\b', lowercase=True, dtype=np.float32)
        counts = self.vec.fit_transform(corpus_texts)
        N = counts.shape[0]
        dfreq = counts.getnnz(axis=0).astype(np.float64)
        self.idf = np.log(1.0 + (N - dfreq + 0.5) / (dfreq + 0.5))
        doc_len = np.asarray(counts.sum(axis=1)).ravel()
        avgdl = max(doc_len.mean(), 1e-9)
        K_d = self.k1 * (1 - self.b + self.b * doc_len / avgdl)
        coo = counts.tocoo()
        den = coo.data.astype(np.float64) + K_d[coo.row]
        newdata = self.idf[coo.col] * (self.k1 + 1.0) * coo.data / den
        self.C = csr_matrix((newdata.astype(np.float32), (coo.row, coo.col)), shape=counts.shape)
        self.N = N
        return self

    def topk(self, query_texts, k):
        nq = len(query_texts)
        idx_out = np.full((nq, k), -1, dtype=np.int32)
        sc_out  = np.zeros((nq, k), dtype=np.float32)
        chunk = 256
        for s in range(0, nq, chunk):
            qc = self.vec.transform(query_texts[s:s+chunk])
            sc = (qc @ self.C.T).toarray()          # (c, N)
            c, N = sc.shape
            kk = min(k, N)
            part = np.argpartition(-sc, kk-1, axis=1)[:, :kk]   
            rows = np.arange(c)[:, None]
            sc_part = sc[rows, part]                             
            
            order = np.argsort(-sc_part, axis=1)                 
            sorted_cols = np.take_along_axis(part, order, axis=1) 
            idx_out[s:s+c, :kk] = sorted_cols
            sc_out[s:s+c, :kk]  = np.take_along_axis(sc_part, order, axis=1)
        return idx_out, sc_out

def build_cat_prior(pos_df: pd.DataFrame) -> dict:
    prior = {}
    sc = pos_df['search_category'].astype(str).values
    ic = pos_df['item_category_id'].astype(str).values
    im = pos_df['item_microcat_id'].astype(str).values
    for s, c, m in zip(sc, ic, im):
        d = prior.setdefault(s, [Counter(), Counter(), 0])
        d[0][c] += 1; d[1][m] += 1; d[2] += 1
    return prior


def cat_channel_for_corpus(prior, item_cat_arr, item_micro_arr, query_sc_arr, k):
    uc, inv_c = np.unique(item_cat_arr, return_inverse=True)
    um, inv_m = np.unique(item_micro_arr, return_inverse=True)
    group_topk = {}
    for sc_key, (cc, cm, tot) in prior.items():
        arr_c = np.log1p(np.array([cc.get(u, 0) for u in uc], dtype=np.float64)) - np.log1p(tot)
        arr_m = np.log1p(np.array([cm.get(u, 0) for u in um], dtype=np.float64)) - np.log1p(tot)
        vec = arr_c[inv_c] + arr_m[inv_m]
        kk = min(k, vec.shape[0])
        part = np.argpartition(-vec, kk-1)[:kk]
        group_topk[sc_key] = part[np.argsort(-vec[part])]
    nq = len(query_sc_arr)
    out = np.full((nq, k), -1, dtype=np.int32)          
    for i, sc_key in enumerate(query_sc_arr):
        lst = group_topk.get(sc_key)
        if lst is not None:
            out[i, :len(lst)] = lst[:k]
    return out


# Opensource модель intfloat/multilingual-e5-base (HuggingFace, разворачивается локально через sentence-transformers), дообучаю её под задачу через losses.MultipleNegativesRankingLoss
def encode(model, texts, bs=128):
    with torch.no_grad():
        return model.encode(texts, batch_size=bs, normalize_embeddings=True,
                            convert_to_numpy=True, show_progress_bar=True).astype('float32')


def dense_topk(item_emb, query_emb, k):
    index = faiss.IndexFlatIP(item_emb.shape[1])
    index.add(np.ascontiguousarray(item_emb))
    _, idx = index.search(np.ascontiguousarray(query_emb), k)
    return idx.astype(np.int32)


def subsample_for_fit(df, steps, bs):
    n = min(steps * bs, len(df))
    return df.sample(n=n, random_state=SEED).reset_index(drop=True)


def train_dense(pairs_df, tag):
    if LOAD_MODEL:
        print(f'[{tag}] загрузка модели из {LOAD_MODEL}')
        m = SentenceTransformer(LOAD_MODEL, device=DEVICE); m.max_seq_length = MAX_SEQ; return m
    if SKIP_DENSE_FIT:
        print(f'[{tag}] dense-fit пропущен, zero-shot {BASE_MODEL}')
        m = SentenceTransformer(BASE_MODEL, device=DEVICE); m.max_seq_length = MAX_SEQ; return m

    examples = [InputExample(texts=[q, p])
                for q, p in zip(pairs_df['q_full'].tolist(), pairs_df['i_full'].tolist())]
    loader = DataLoader(examples, shuffle=True, batch_size=TRAIN_BS, num_workers=0, drop_last=True)
    model = SentenceTransformer(BASE_MODEL, device=DEVICE)
    model.max_seq_length = MAX_SEQ
    loss = losses.MultipleNegativesRankingLoss(model)
    print(f'[{tag}] fit: {len(examples):,} пар, ~{len(loader):,} шагов, bs={TRAIN_BS}, lr={TRAIN_LR}')
    model.fit(train_objectives=[(loader, loss)], epochs=1,
              warmup_steps=max(50, int(len(loader)*0.1)),
              optimizer_params={'lr': TRAIN_LR}, show_progress_bar=True, use_amp=True)
    if SAVE_MODEL:
        p = os.path.join(WORK_DIR, f'model_{tag}'); model.save(p); print(f'[{tag}] сохранено: {p}')
    del examples, loader, loss; gc.collect(); torch.cuda.empty_cache()
    return model

# сборка каналов

def compute_channels(queries_df, items_df, model, prior, k=CHANNEL_K):
    bm25 = BM25().fit(items_df['i_bow'].tolist())
    bm_idx, _ = bm25.topk(queries_df['q_bow'].tolist(), k)
    del bm25; gc.collect()

    item_emb  = encode(model, items_df['i_full'].tolist())
    query_emb = encode(model, queries_df['q_full'].tolist())
    dn_idx = dense_topk(item_emb, query_emb, k)
    del item_emb, query_emb; gc.collect(); torch.cuda.empty_cache()

    ct_idx = cat_channel_for_corpus(
        prior,
        items_df['item_category_id'].astype(str).values,
        items_df['item_microcat_id'].astype(str).values,
        queries_df['search_category'].astype(str).values,
        k)

    q_deliv = (pd.to_numeric(queries_df['search_is_delivery_search'], errors='coerce')  # ФИКС 1
                 .fillna(0).astype(int).values)

    return {'bm': bm_idx, 'dn': dn_idx, 'ct': ct_idx,
            'item_loc': items_df['item_location_id'].astype(str).values,
            'q_loc':    queries_df['search_location_id'].astype(str).values,
            'q_deliv':  q_deliv}


# RRF 
def fuse(ch, params, top=TOP_K):
    bm, dn, ct = ch['bm'], ch['dn'], ch['ct']
    item_loc, q_loc, q_deliv = ch['item_loc'], ch['q_loc'], ch['q_deliv']
    nq = bm.shape[0]
    preds = []
    for i in range(nq):
        score = {}
        for arr, kk in ((bm[i], params['k_bm']), (dn[i], params['k_dn'])):
            for r, pos in enumerate(arr):
                if pos >= 0:
                    score[pos] = score.get(pos, 0.0) + 1.0 / (kk + r + 1)
        if params['use_ct']:
            for r, pos in enumerate(ct[i]):
                if pos >= 0:
                    score[pos] = score.get(pos, 0.0) + 1.0 / (params['k_ct'] + r + 1)
        if params['loc_w'] > 0 and score:
            cand = np.fromiter(score.keys(), dtype=np.int64, count=len(score))
            bonus = ((item_loc[cand] == q_loc[i]) & (q_deliv[i] == 0))
            for c, b in zip(cand, bonus):
                if b:
                    score[c] += params['loc_w']
        items = sorted(score.items(), key=lambda x: -x[1])[:top]
        preds.append([p for p, _ in items])
    return preds


def recall_at_k(preds_ids, positives_list):
    vals = []
    for pred, pos in zip(preds_ids, positives_list):
        if not pos:
            continue
        vals.append(len(set(pred) & pos) / len(pos))
    return float(np.mean(vals)) if vals else 0.0



print('[load] читаем parquet')
train = pd.read_parquet(f'{DATA_DIR}/train.parquet')
bq    = pd.read_parquet(f'{DATA_DIR}/benchmark_queries.parquet')
bi    = pd.read_parquet(f'{DATA_DIR}/benchmark_items.parquet')

print('[text] векторизованная сборка текстов')
add_query_cols(train); add_item_cols(train)
add_query_cols(bq);    add_item_cols(bi)

sig_cols = ['search_query', 'search_location_id', 'search_category',
            'search_infm_params_text', 'search_is_delivery_search']
train['_sig'] = train[sig_cols].astype(str).agg('||'.join, axis=1)
uniq_sigs = train['_sig'].unique()
np.random.shuffle(uniq_sigs)
n_val = max(1, int(len(uniq_sigs) * VAL_FRAC))
val_sigs = set(uniq_sigs[:n_val]); tr_sigs = set(uniq_sigs[n_val:])
print(f'[split] уникальных запросов={len(uniq_sigs)}, val-групп={len(val_sigs)}')

best_params = dict(k_bm=60, k_dn=60, k_ct=40, use_ct=True, loc_w=0.03)

if USE_VAL:
    print(f'[val] dense-fit на подвыборке tr-train (FAST_STEPS={FAST_STEPS})...')
    tr_part = train[train['_sig'].isin(tr_sigs)].reset_index(drop=True)
    va_part = train[train['_sig'].isin(val_sigs)].reset_index(drop=True)
    model_val = train_dense(subsample_for_fit(tr_part, FAST_STEPS, TRAIN_BS), 'val')
    prior_val = build_cat_prior(tr_part)

    val_items = train.drop_duplicates('item_id').reset_index(drop=True)
    for c in ['item_category_id','item_microcat_id','item_title_raw','item_infm_params_text',
              'item_description_raw','item_location_id']:
        if c not in val_items.columns:
            val_items[c] = ''
    add_item_cols(val_items)

    pos_map = {s: set(g['item_id'].astype(str)) for s, g in va_part.groupby('_sig')}
    val_queries = va_part.drop_duplicates('_sig').reset_index(drop=True)
    val_queries = val_queries[val_queries['_sig'].isin(pos_map)].reset_index(drop=True)
    pos_list = [pos_map[s] for s in val_queries['_sig']]

    print('[val] расчёт каналов...')
    ch_val = compute_channels(val_queries, val_items, model_val, prior_val)
    val_item_ids = val_items['item_id'].astype(str).to_numpy()

    tune_idx = np.random.choice(len(val_queries), size=min(TUNE_QUERIES, len(val_queries)), replace=False)
    tune_ch = dict(ch_val)
    for kk in ('bm', 'dn', 'ct', 'q_loc', 'q_deliv'):  
        tune_ch[kk] = ch_val[kk][tune_idx]
    tune_pos = [pos_list[i] for i in tune_idx]

    grid = []
    for use_ct in (True, False):
        for k_ct in (20, 40, 60):
            for loc_w in (0.0, 0.03, 0.06):
                grid.append(dict(k_bm=60, k_dn=60, k_ct=k_ct, use_ct=use_ct, loc_w=loc_w))
    best_score = -1.0
    for p in grid:
        pr = fuse(tune_ch, p)
        pr_ids = [[val_item_ids[i] for i in row] for row in pr]
        sc = recall_at_k(pr_ids, tune_pos)
        if sc > best_score:
            best_score, best_params = sc, p
        print(f'   params={p} -> Recall@50(tune)={sc:.4f}')

    pr_all = fuse(ch_val, best_params)
    pr_all_ids = [[val_item_ids[i] for i in row] for row in pr_all]
    full_val = recall_at_k(pr_all_ids, pos_list)
    print(f'[val] ЛУЧШИЕ params={best_params}')
    print(f'[val] Recall@50 по tune-подвыборке = {best_score:.4f}')
    print(f'[val] Recall@50 по ВСЕЙ валидации ({len(pos_list)} запросов) = {full_val:.4f}')
    del model_val, ch_val, val_items; gc.collect(); torch.cuda.empty_cache()

print(f'[final] dense-fit на подвыборке train (FAST_STEPS={FAST_STEPS})...')
model_full = train_dense(subsample_for_fit(train, FAST_STEPS, TRAIN_BS), 'full')
prior_full = build_cat_prior(train) 

print('[final] расчёт каналов на benchmark_items...')
ch_test = compute_channels(bq, bi, model_full, prior_full)
print(f'[final] фьюз params={best_params} ...')
preds = fuse(ch_test, best_params)
bi_ids = bi['item_id'].astype(str).to_numpy()

rows, seen_q = [], set()
for qid, row in zip(bq['query_id'].astype(str).values, preds):
    if qid in seen_q:
        continue
    seen_q.add(qid)
    ids = [bi_ids[i] for i in row if 0 <= i < len(bi_ids)]
    rows.append({'query_id': qid, 'answer': ' '.join(list(dict.fromkeys(ids))[:TOP_K])})

missing = set(bq['query_id'].astype(str)) - seen_q          
if missing:
    print(f'[WARN] {len(missing)} query_id без предсказания -> пустой answer')
    for qid in missing:
        rows.append({'query_id': qid, 'answer': ''})

answer = pd.DataFrame(rows, columns=['query_id', 'answer'])
out = os.path.join(WORK_DIR, 'answer.csv')
answer.to_csv(out, index=False, encoding='utf-8')
print(f'\n[done] {out}  строк={len(answer)}  (ожидаемо {bq["query_id"].nunique()})')
print(answer.head(2).to_string())

/tmp/ipykernel_58/1086368308.py:16: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers import SentenceTransformer, InputExample, losses


[load] читаем parquet...
[text] векторизованная сборка текстов...
[split] уникальных запросов=354463, val-групп=35446
[val] dense-fit на подвыборке tr-train (FAST_STEPS=2000)...


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[full] сохранено: /kaggle/working/model_full
[final] расчёт каналов на benchmark_items...


Batches:   0%|          | 0/1479 [00:00<?, ?it/s]

Batches:   0%|          | 0/20 [00:00<?, ?it/s]

[final] фьюз params={'k_bm': 60, 'k_dn': 60, 'k_ct': 20, 'use_ct': False, 'loc_w': 0.06} ...

[done] /kaggle/working/answer.csv  строк=2452  (ожидаемо 2452)
           query_id                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                        